In [1]:
# ============================================
# CELDA 1: IMPORTAR LIBRERÍAS
# ============================================
# pandas: manejo de datos en tablas
# numpy: apoyo para cálculos numéricos
# matplotlib y seaborn: gráficos
# scipy.stats: chi-cuadrado para V de Cramér

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency

# Configuración visual general de los gráficos
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

In [2]:
# ============================================
# CELDA 2: CARGAR UN CSV PÚBLICO
# ============================================
# Usaremos el dataset Titanic en formato CSV, disponible públicamente.
# Este dataset tiene variables numéricas y categóricas, por lo que sirve
# para calcular Pearson, Spearman y V de Cramér.

url = "https://github.com/mwaskom/seaborn-data/raw/master/titanic.csv"

df = pd.read_csv(url)

# Mostrar las primeras filas para verificar la carga
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
# ============================================
# CELDA 3: EXPLORACIÓN INICIAL
# ============================================
# Revisamos dimensiones, tipos de datos y valores nulos.

print("Dimensiones del dataset:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum())

Dimensiones del dataset: (891, 15)

Tipos de datos:
survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object

Valores nulos por columna:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [4]:
# ============================================
# CELDA 4: RESUMEN ESTADÍSTICO
# ============================================
# describe() entrega medidas básicas para variables numéricas:
# cantidad, media, desviación estándar, mínimos, cuartiles y máximos.

df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:
# ============================================
# CELDA 5: PREPARACIÓN DE VARIABLES
# ============================================
# Pearson y Spearman se aplican a variables numéricas.
# V de Cramér se aplica a variables categóricas.
#
# En Titanic, "pclass" aparece como numérica, pero conceptualmente
# representa categorías (1ra, 2da, 3ra clase). Por eso la convertimos
# a categórica para el análisis de Cramér.

df["pclass_cat"] = df["pclass"].astype("category")
df["survived_cat"] = df["survived"].astype("category")
df["adult_male"] = df["adult_male"].astype(str)
df["alone"] = df["alone"].astype(str)

# Selección de variables numéricas para correlaciones
columnas_numericas = ["survived", "age", "sibsp", "parch", "fare"]

# Selección de variables categóricas para V de Cramér
columnas_categoricas = ["survived_cat", "pclass_cat", "sex", "embarked", "class", "who", "adult_male", "alone"]

# Crear una copia solo con las columnas elegidas
datos = df[columnas_numericas + columnas_categoricas].copy()

# Rellenar nulos en categóricas con una etiqueta legible
for col in columnas_categoricas:
    # Añadir 'Desconocido' como una categoría válida antes de usar fillna
    if datos[col].dtype == 'category':
        if 'Desconocido' not in datos[col].cat.categories:
            datos[col] = datos[col].cat.add_categories('Desconocido')
    datos[col] = datos[col].fillna("Desconocido")

print("Columnas numéricas:", columnas_numericas)
print("Columnas categóricas:", columnas_categoricas)

In [ ]:
# ============================================
# CELDA 6: CORRELACIÓN DE PEARSON
# ============================================
# Pearson mide la relación lineal entre variables numéricas.
# Su valor va entre -1 y 1.

pearson_corr = datos[columnas_numericas].corr(method="pearson")

print("Matriz de correlación de Pearson:")
pearson_corr

In [ ]:
# ============================================
# CELDA 7: CORRELACIÓN DE SPEARMAN
# ============================================
# Spearman mide la relación monotónica entre variables numéricas,
# usando rangos en lugar de los valores originales.
# También toma valores entre -1 y 1.

spearman_corr = datos[columnas_numericas].corr(method="spearman")

print("Matriz de correlación de Spearman:")
spearman_corr

In [ ]:
# ============================================
# CELDA 8: FUNCIÓN PARA V DE CRAMÉR
# ============================================
# V de Cramér mide la asociación entre dos variables categóricas.
# Se basa en el estadístico chi-cuadrado.
# Su valor va entre 0 y 1:
# 0  -> sin asociación
# 1  -> asociación fuerte

def cramers_v(x, y):
    # Construir tabla de contingencia
    tabla = pd.crosstab(x, y)

    # Calcular chi-cuadrado
    chi2 = chi2_contingency(tabla)[0]

    # Número total de observaciones
    n = tabla.to_numpy().sum()

    # Número mínimo de categorías menos 1
    k = min(tabla.shape[0], tabla.shape[1])

    # Evitar división por cero
    if n == 0 or k <= 1:
        return np.nan

    # Fórmula de V de Cramér
    return np.sqrt(chi2 / (n * (k - 1)))

In [ ]:
# ============================================
# CELDA 9: MATRIZ DE V DE CRAMÉR
# ============================================
# Calculamos la asociación entre todas las variables categóricas seleccionadas.

cramer_matrix = pd.DataFrame(
    np.zeros((len(columnas_categoricas), len(columnas_categoricas))),
    index=columnas_categoricas,
    columns=columnas_categoricas
)

for col1 in columnas_categoricas:
    for col2 in columnas_categoricas:
        cramer_matrix.loc[col1, col2] = cramers_v(datos[col1], datos[col2])

print("Matriz de V de Cramér:")
cramer_matrix

In [ ]:
# ============================================
# CELDA 10: HISTOGRAMAS
# ============================================
# Permiten ver la distribución de cada variable numérica.

datos[columnas_numericas].hist(bins=20, figsize=(14, 10))
plt.suptitle("Histogramas de variables numéricas", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELDA 11: BOXPLOTS
# ============================================
# Permiten ver mediana, dispersión y posibles valores atípicos.

for col in columnas_numericas:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=datos[col])
    plt.title(f"Boxplot de {col}")
    plt.show()

In [ ]:
# ============================================
# CELDA 12: SCATTER PLOT
# ============================================
# Relacionamos edad y tarifa pagada.
# Además, se colorea por supervivencia.

plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="age", y="fare", hue="survived")
plt.title("Dispersión entre edad y tarifa pagada")
plt.xlabel("Edad")
plt.ylabel("Tarifa")
plt.show()

In [ ]:
# ============================================
# CELDA 13: GRÁFICO DE BARRAS
# ============================================
# Mostramos cuántos pasajeros hay por sexo.

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="sex")
plt.title("Cantidad de pasajeros por sexo")
plt.xlabel("Sexo")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# ============================================
# CELDA 14: BOXPLOT DE FARE SEGÚN CLASE
# ============================================
# Permite comparar la distribución de la tarifa entre clases.

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="class", y="fare")
plt.title("Distribución de tarifa según clase")
plt.xlabel("Clase")
plt.ylabel("Tarifa")
plt.show()

In [ ]:
# ============================================
# CELDA 15: HEATMAP DE PEARSON
# ============================================
# El mapa de calor permite ver rápidamente qué pares de variables
# tienen relación lineal positiva o negativa.

plt.figure(figsize=(8, 6))
sns.heatmap(pearson_corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Mapa de calor - Correlación de Pearson")
plt.show()

In [ ]:
# ============================================
# CELDA 16: HEATMAP DE SPEARMAN
# ============================================
# Este mapa de calor resume relaciones monotónicas.

plt.figure(figsize=(8, 6))
sns.heatmap(spearman_corr, annot=True, cmap="viridis", fmt=".2f")
plt.title("Mapa de calor - Correlación de Spearman")
plt.show()

In [ ]:
# ============================================
# CELDA 17: HEATMAP DE V DE CRAMÉR
# ============================================
# Resume la intensidad de asociación entre variables categóricas.

plt.figure(figsize=(10, 8))
sns.heatmap(cramer_matrix, annot=True, cmap="YlOrBr", fmt=".2f")
plt.title("Mapa de calor - V de Cramér")
plt.show()

In [ ]:
# ============================================
# CELDA 18: PARES CON MAYOR CORRELACIÓN
# ============================================
# Esta celda no calcula nuevas correlaciones.
# Solo muestra los pares cuya correlación supera un umbral.
# Si no aparece nada, significa que ningún par alcanzó ese valor.

umbral = 0.20

print(f"Pares con correlación alta en Pearson (|r| >= {umbral}):")
encontro = False
for i in range(len(pearson_corr.columns)):
    for j in range(i + 1, len(pearson_corr.columns)):
        valor = pearson_corr.iloc[i, j]
        if abs(valor) >= umbral:
            print(f"{pearson_corr.index[i]} vs {pearson_corr.columns[j]}: {valor:.2f}")
            encontro = True

if not encontro:
    print("No se encontraron pares que superen el umbral.")

print(f"\nPares con correlación alta en Spearman (|r| >= {umbral}):")
encontro = False
for i in range(len(spearman_corr.columns)):
    for j in range(i + 1, len(spearman_corr.columns)):
        valor = spearman_corr.iloc[i, j]
        if abs(valor) >= umbral:
            print(f"{spearman_corr.index[i]} vs {spearman_corr.columns[j]}: {valor:.2f}")
            encontro = True

if not encontro:
    print("No se encontraron pares que superen el umbral.")

In [ ]:
# ============================================
# CELDA 19: PARES CON MAYOR ASOCIACIÓN CATEGÓRICA
# ============================================
# Mostramos pares de variables con V de Cramér relativamente alta.

print("Pares con asociación alta en V de Cramér (>= 0.30):")
for i in range(len(cramer_matrix.columns)):
    for j in range(i + 1, len(cramer_matrix.columns)):
        valor = cramer_matrix.iloc[i, j]
        if valor >= 0.30:
            print(f"{cramer_matrix.index[i]} vs {cramer_matrix.columns[j]}: {valor:.2f}")